# 📊 SonDatos — Pipeline de Sentimiento · **Instagram**
### Estudio replicable para cualquier personaje o marca

**Diferencia clave con YouTube:** Instagram no ofrece una API pública gratuita para leer comentarios de cuentas ajenas. Por eso este notebook trabaja en dos etapas:

**Etapa A — Conseguir los comentarios (fuera de este notebook).** Opciones, de más a menos recomendada:
1. **Cuenta propia o de un cliente que te autorice:** la Graph API de Meta permite exportar comentarios de las publicaciones de una cuenta *business/creator* que administres. Es la vía oficial y estable.
2. **Servicios de extracción (Apify, Bright Data, etc.):** exportan comentarios de posts públicos a CSV/JSON por un costo bajo por corrida. *Advertencia SUBROSA:* estos servicios operan en zona gris respecto a los términos de servicio de Instagram — evalúa el riesgo reputacional/legal según el cliente y el uso, y nunca los uses con cuentas personales tuyas logueadas.
3. **Exportación manual asistida** para muestras pequeñas.

**Etapa B — Este notebook:** le das el CSV con los comentarios y ejecuta el mismo análisis del pipeline de YouTube: relevancia, aspectos, anti-bot, sentimiento con GPU, series temporales, impacto de eventos, gráficos de marca y muestra de validación.

**Formato del CSV de entrada** (columnas mínimas):
| columna | contenido |
|---|---|
| `text` | texto del comentario |
| `published_at` | fecha (ISO: 2026-07-01T14:30:00 o similar) |
| `author` | usuario que comenta |
| `post_id` | identificador o URL del post |

El notebook detecta y renombra automáticamente las variantes comunes de los exports de Apify (`ownerUsername`, `timestamp`, etc.).

**Antes de correr:** activa GPU (Entorno de ejecución → T4) y edita solo la celda ⚙️.

*SonDatos · división de análisis de SUBROSA · v1.0 · jul 2026*

In [ ]:
#@title ⚙️ CONFIGURACIÓN — edita esto y nada más { display-mode: "form" }
NOMBRE_ESTUDIO = "mi-estudio-ig"
SUJETO         = "Nombre del personaje o marca"

# Si el CSV trae comentarios de posts ya seleccionados sobre el tema,
# puedes dejar FILTRAR_RELEVANCIA en False (todos cuentan).
FILTRAR_RELEVANCIA = True
KEYWORDS_SUJETO    = ["keyword1", "keyword2"]

ASPECTOS = {
    "aspecto_1": ["keyword", "keyword"],
    "aspecto_2": ["keyword", "keyword"],
}

EVENTOS = [
    {"fecha": "2026-01-15", "nombre": "Evento 1"},
    {"fecha": "2026-03-02", "nombre": "Evento 2"},
]

FECHA_INICIO = "2025-01-01"
FECHA_FIN    = "2026-12-31"

TEMA = "claro"   # "claro" u "oscuro"
MARCA = {
    "nombre": "SonDatos", "url": "sondatos.do",
    "oscuro": {"fondo":"#421034","pos":"#E1407A","neg":"#6B2D8B",
               "neutro":"#9A8496","acento":"#F3A0C0","texto":"#F5EDF2"},
    "claro":  {"fondo":"#FFFFFF","pos":"#D6336C","neg":"#6B2D8B",
               "neutro":"#8A7A85","acento":"#B03070","texto":"#2A1A24"},
}

AGREGACION = "W"
VENTANA_EVENTO_DIAS = 14
BOOTSTRAP_ITER = 2000
MUESTRA_VALIDACION = 400
print("Configuración cargada ✓")

In [ ]:
#@title 1 · Instalación
!pip install -q pysentimiento pyarrow
print("Listo ✓")

In [ ]:
#@title 2 · Subir el CSV de comentarios
from google.colab import files
import pandas as pd, unicodedata

up = files.upload()
fname = list(up.keys())[0]
raw = pd.read_csv(fname) if fname.endswith(".csv") else pd.read_json(fname)
print(f"Archivo: {fname} · {len(raw):,} filas")
print("Columnas:", list(raw.columns))

# normalizar nombres de columnas de exports comunes (Apify y similares)
ALIAS = {
    "text": ["text","comment","comment_text","message","body"],
    "published_at": ["published_at","timestamp","created_at","createdAt","date","taken_at"],
    "author": ["author","ownerUsername","username","user","owner_username"],
    "post_id": ["post_id","postUrl","post_url","media_id","shortcode","url"],
    "like_count": ["like_count","likesCount","likes","likeCount"],
}
ren = {}
for destino, opciones in ALIAS.items():
    for o in opciones:
        if o in raw.columns: ren[o] = destino; break
raw = raw.rename(columns=ren)
faltan = [c for c in ["text","published_at"] if c not in raw.columns]
assert not faltan, f"Faltan columnas obligatorias: {faltan} — renómbralas en tu CSV"
for c in ["author","post_id"]:
    if c not in raw.columns: raw[c] = ""
if "like_count" not in raw.columns: raw["like_count"] = 0
print("Columnas normalizadas ✓")

In [ ]:
#@title 3 · Limpieza, relevancia, aspectos y anti-bot
def norm(t):
    t = unicodedata.normalize("NFKD", str(t).lower())
    return "".join(c for c in t if not unicodedata.combining(c))

df = raw.copy()
df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
df = df.dropna(subset=["published_at","text"])
df = df[(df["published_at"] >= pd.Timestamp(FECHA_INICIO, tz="UTC")) &
        (df["published_at"] <= pd.Timestamp(FECHA_FIN, tz="UTC"))]
df["text_norm"] = df["text"].map(norm)
df["text_model"] = (df["text"].astype(str)
                    .str.replace(r"https?://\S+","",regex=True)
                    .str.replace(r"@[\w.-]+","@usuario",regex=True)
                    .str.replace(r"\s+"," ",regex=True).str.strip())
df["comment_id"] = range(len(df))

if FILTRAR_RELEVANCIA:
    KW = [norm(k) for k in KEYWORDS_SUJETO]
    df["relevant"] = df["text_norm"].map(lambda t: any(k in t for k in KW)) & (df["text_model"].str.len() >= 8)
else:
    df["relevant"] = df["text_model"].str.len() >= 8

ASP = {a: [norm(k) for k in kws] for a, kws in ASPECTOS.items()}
def aspecto(t):
    hits = [a for a, kws in ASP.items() if any(k in t for k in kws)]
    return hits[0] if len(hits) == 1 else ("mixto" if len(hits) > 1 else "general")
df["aspect"] = df["text_norm"].map(aspecto)

dup = df.groupby("text_norm")["comment_id"].transform("count") > 3
vacio = df["text_norm"].map(lambda t: len(set(c for c in t if c.isalpha())) < 4)
hiper = df.groupby(["post_id","author"])["comment_id"].transform("count") > 10
df["bot_flag"] = dup | vacio | hiper

util = df[df["relevant"] & ~df["bot_flag"]]
df.to_parquet("comments_clean.parquet", index=False)
print(f"Crudos: {len(raw):,} | Relevantes: {df['relevant'].sum():,} | "
      f"Spam/bot: {df['bot_flag'].sum():,} | UTILIZABLES: {len(util):,}")
print(util["aspect"].value_counts().to_string())

In [ ]:
#@title 4 · Sentimiento (robertuito, GPU)
from pysentimiento import create_analyzer
analyzer = create_analyzer(task="sentiment", lang="es")

work = df[df["relevant"] & ~df["bot_flag"]].copy().reset_index(drop=True)
texts = work["text_model"].fillna("").tolist()
labels, probs = [], []
B = 256
for i in range(0, len(texts), B):
    for r in analyzer.predict(texts[i:i+B]):
        labels.append(r.output)
        probs.append((r.probas.get("POS",0), r.probas.get("NEU",0), r.probas.get("NEG",0)))
    if (i//B) % 10 == 0: print(f"  {min(i+B,len(texts)):,}/{len(texts):,}")

work["sentiment"] = labels
work[["p_pos","p_neu","p_neg"]] = pd.DataFrame(probs, index=work.index)
work["confidence"] = work[["p_pos","p_neu","p_neg"]].max(axis=1)
work.to_parquet("comments_scored.parquet", index=False)
print("\nDistribución:", work["sentiment"].value_counts(normalize=True).round(3).to_dict())

n = MUESTRA_VALIDACION
sample = (work.groupby(["sentiment","aspect"], group_keys=False)
          .apply(lambda g: g.sample(min(len(g), max(1, int(n*len(g)/max(len(work),1)))), random_state=33))
          .head(n))
val = sample[["comment_id","text","sentiment","aspect","confidence"]].copy()
val["etiqueta_manual"] = ""; val["postura"] = ""
val.to_csv("sample_to_label.csv", index=False, encoding="utf-8-sig")
print(f"Muestra de validación: sample_to_label.csv ({len(val)})")

In [ ]:
#@title 5 · Análisis: series, IC bootstrap e impacto de eventos
import numpy as np
from scipy import stats
RNG = np.random.default_rng(33)

def net_ci(labels, n_boot=BOOTSTRAP_ITER):
    arr = labels.map({"POS":1,"NEU":0,"NEG":-1}).to_numpy()
    if len(arr) < 5: return (arr.mean() if len(arr) else np.nan), np.nan, np.nan
    boots = RNG.choice(arr, size=(n_boot, len(arr)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return arr.mean(), lo, hi

def serie(d):
    rows = []
    for w, g in d.groupby(pd.Grouper(key="published_at", freq=AGREGACION)):
        if g.empty: continue
        p, lo, hi = net_ci(g["sentiment"])
        rows.append({"week": w, "n": len(g), "net": p, "lo": lo, "hi": hi})
    return pd.DataFrame(rows)

weekly = serie(work); weekly.to_csv("serie_semanal.csv", index=False)
por_aspecto = {a: serie(g) for a, g in work.groupby("aspect")}

win = pd.Timedelta(days=VENTANA_EVENTO_DIAS)
imp = []
for ev in EVENTOS:
    d = pd.Timestamp(ev["fecha"], tz="UTC")
    pre  = work[(work["published_at"] >= d-win) & (work["published_at"] < d)]
    post = work[(work["published_at"] >= d) & (work["published_at"] < d+win)]
    row = {"evento": ev["nombre"], "fecha": ev["fecha"], "n_pre": len(pre), "n_post": len(post)}
    if len(pre) >= 20 and len(post) >= 20:
        a, b = (pre["sentiment"]=="NEG").sum(), (post["sentiment"]=="NEG").sum()
        pp = (a+b)/(len(pre)+len(post))
        se = np.sqrt(pp*(1-pp)*(1/len(pre)+1/len(post)))
        z = ((b/len(post))-(a/len(pre)))/se if se>0 else np.nan
        row.update({"neg_pre": a/len(pre), "neg_post": b/len(post),
                    "delta_pp": (b/len(post)-a/len(pre))*100,
                    "p_value": 2*(1-stats.norm.cdf(abs(z))),
                    "delta_net": net_ci(post["sentiment"],500)[0]-net_ci(pre["sentiment"],500)[0]})
    imp.append(row)
impacto = pd.DataFrame(imp); impacto.to_csv("impacto_eventos.csv", index=False)

g_net, g_lo, g_hi = net_ci(work["sentiment"])
print(f"NET SENTIMENT GLOBAL: {g_net:+.3f} (IC95 [{g_lo:+.3f},{g_hi:+.3f}]) · n={len(work):,}")
for a in ASPECTOS:
    s = work[work["aspect"]==a]
    if len(s):
        p,lo,hi = net_ci(s["sentiment"])
        print(f"  {a:20s}: {p:+.3f} (IC95 [{lo:+.3f},{hi:+.3f}]) · n={len(s):,}")
print("\nImpacto de eventos:")
print(impacto.round(3).to_string(index=False))

In [ ]:
#@title 6 · Gráficos con tu marca
import matplotlib.pyplot as plt, matplotlib.dates as mdates
C = MARCA["oscuro" if TEMA == "oscuro" else "claro"]
FOOT = f"Fuente: comentarios públicos de Instagram · Modelo: robertuito · {MARCA['nombre']} — {MARCA['url']}"

def estilo(ax, fig):
    fig.patch.set_facecolor(C["fondo"]); ax.set_facecolor(C["fondo"])
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    for s in ["left","bottom"]: ax.spines[s].set_color(C["neutro"])
    ax.tick_params(colors=C["texto"]); ax.grid(True, alpha=.22, color=C["neutro"])
    ax.yaxis.label.set_color(C["texto"])

def eventos_v(ax):
    ym = ax.get_ylim()[1]
    for ev in EVENTOS:
        d = pd.Timestamp(ev["fecha"])
        ax.axvline(d, color=C["acento"], alpha=.55, lw=1, ls="--")
        ax.annotate(ev["nombre"], xy=(d, ym), fontsize=7.5, color=C["acento"],
                    rotation=90, va="top", ha="right")

def guardar(fig, nombre):
    fig.text(.5,.01,FOOT,ha="center",fontsize=7.5,color=C["neutro"])
    fig.savefig(f"{nombre}.png", dpi=150, bbox_inches="tight", facecolor=C["fondo"])
    plt.close(fig); print(f"  {nombre}.png ✓")

wk = weekly.copy(); wk["week"] = pd.to_datetime(wk["week"])
fig, ax = plt.subplots(figsize=(12,6)); estilo(ax,fig)
ax.fill_between(wk["week"], wk["lo"], wk["hi"], color=C["pos"], alpha=.13, label="IC 95%")
ax.plot(wk["week"], wk["net"], color=C["pos"], lw=2.2, label="Net sentiment")
ax.axhline(0, color=C["neutro"], lw=.8); eventos_v(ax)
ax.set_title(f"Sentimiento hacia {SUJETO} en Instagram", color=C["texto"], fontsize=15, fontweight="bold", pad=14)
leg = ax.legend(loc="lower left", frameon=False); [t.set_color(C["texto"]) for t in leg.get_texts()]
guardar(fig, "01_timeline")

fig, ax = plt.subplots(figsize=(12,6)); estilo(ax,fig)
paleta = [C["pos"], C["neg"], C["acento"], C["neutro"]]
i = 0
for a, s in por_aspecto.items():
    if a in ("mixto","general") or s.empty: continue
    s = s.copy(); s["week"] = pd.to_datetime(s["week"])
    ax.plot(s["week"], s["net"].rolling(2, min_periods=1).mean(),
            color=paleta[i % 4], lw=2.2, label=a); i += 1
if i:
    ax.axhline(0, color=C["neutro"], lw=.8)
    ax.set_title("Net sentiment por aspecto", color=C["texto"], fontsize=15, fontweight="bold", pad=14)
    leg = ax.legend(loc="lower left", frameon=False); [t.set_color(C["texto"]) for t in leg.get_texts()]
    guardar(fig, "02_aspectos")
else:
    plt.close(fig)

ii = impacto.dropna(subset=["delta_net"]) if "delta_net" in impacto else pd.DataFrame()
if not ii.empty:
    fig, ax = plt.subplots(figsize=(11,5.5)); estilo(ax,fig)
    cols = [C["neg"] if d < 0 else C["pos"] for d in ii["delta_net"]]
    bars = ax.barh(ii["evento"], ii["delta_net"], color=cols)
    for b, (_, r) in zip(bars, ii.iterrows()):
        sig = " *" if r.get("p_value",1) < .05 else ""
        ax.text(b.get_width(), b.get_y()+b.get_height()/2, f' {r["delta_net"]:+.2f}{sig}',
                va="center", fontsize=9, color=C["texto"])
    ax.axvline(0, color=C["neutro"], lw=.8)
    ax.set_title("Cambio de sentimiento tras cada evento", color=C["texto"], fontsize=15, fontweight="bold", pad=14)
    ax.set_xlabel("Δ net sentiment (post − pre) · * p<0.05", color=C["texto"])
    guardar(fig, "03_eventos")

fig, ax = plt.subplots(figsize=(12,5)); estilo(ax,fig)
ax.bar(wk["week"], wk["n"], width=6, color=C["acento"], alpha=.9); eventos_v(ax)
ax.set_title("Volumen de conversación", color=C["texto"], fontsize=15, fontweight="bold", pad=14)
guardar(fig, "04_volumen")

In [ ]:
#@title 7 · Descargar todo (ZIP)
import zipfile
zname = f"{NOMBRE_ESTUDIO}_resultados.zip"
with zipfile.ZipFile(zname, "w") as z:
    for f in ["comments_clean.parquet","comments_scored.parquet","serie_semanal.csv",
              "impacto_eventos.csv","sample_to_label.csv",
              "01_timeline.png","02_aspectos.png","03_eventos.png","04_volumen.png"]:
        try: z.write(f)
        except FileNotFoundError: pass
from google.colab import files
files.download(zname)
print(f"ZIP listo: {zname}")

## 8 · (Opcional) Validación humana y corrección de sesgo
Mismo procedimiento que en el pipeline de YouTube: etiqueta `sample_to_label.csv`
(**etiqueta_manual**: POS/NEU/NEG por tono, a ciegas · **postura**: a_favor/en_contra/no_clara)
y ejecuta la celda siguiente subiendo el archivo.

In [ ]:
#@title 8 · Subir CSV etiquetado → validación + corrección
from google.colab import files as gfiles
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

up = gfiles.upload()
lab = pd.read_csv(list(up.keys())[0])
sc = pd.read_parquet("comments_scored.parquet")

m = lab.merge(sc[["comment_id","sentiment"]].rename(columns={"sentiment":"modelo"}),
              on="comment_id", how="left").dropna(subset=["modelo"])
v = m[m["etiqueta_manual"].isin(["POS","NEU","NEG"])]
print(f"Pares válidos: {len(v)}\n")

L = ["POS","NEU","NEG"]
cm = confusion_matrix(v["etiqueta_manual"], v["modelo"], labels=L)
print(pd.DataFrame(cm, index=L, columns=L).to_string())
print(f"\nAccuracy: {accuracy_score(v['etiqueta_manual'], v['modelo']):.3f} | "
      f"F1 macro: {f1_score(v['etiqueta_manual'], v['modelo'], average='macro'):.3f}\n")
print(classification_report(v["etiqueta_manual"], v["modelo"], digits=3))

A = cm.T.astype(float); A = A / A.sum(axis=0, keepdims=True)
def corrige(dist):
    p = np.array([dist.get(l,0) for l in L])
    q = np.linalg.solve(A, p); q = np.clip(q, 0, None); return q/q.sum()
d = sc["sentiment"].value_counts(normalize=True)
q = corrige(d)
print(f"GLOBAL modelo   : POS {d.get('POS',0):.1%} NEU {d.get('NEU',0):.1%} NEG {d.get('NEG',0):.1%} -> net {d.get('POS',0)-d.get('NEG',0):+.3f}")
print(f"GLOBAL corregido: POS {q[0]:.1%} NEU {q[1]:.1%} NEG {q[2]:.1%} -> net {q[0]-q[2]:+.3f}")

if "postura" in v.columns and v["postura"].notna().any():
    print("\nCruce sentimiento del modelo × postura manual (%):")
    print((pd.crosstab(v["modelo"], v["postura"], normalize="index")*100).round(1).to_string())